# Katube Colab - Download de Áudio do YouTube",

    "Sistema modular para download de áudio com persistência no Google Drive"

In [ ]:
# Clone do repositório
!git clone https://github.com/seu-usuario/katube-colab.git
%cd katube-colab

In [ ]:
# Instala dependências
!pip install -q -r requirements.txt

##  Monta Google Drive

In [ ]:
from downloaders import DriveManager
from config import Config

# Monta Drive
drive_manager = DriveManager()
drive_manager.mount_drive()

# Cria estrutura de pastas
drive_manager.setup_folder_structure()

# Verifica espaço
space_info = drive_manager.check_drive_space()
if space_info['available']:
    print(f"Espaço livre no Drive: {space_info['free_formatted']}")

## Configuração (Opcional)

Personalize as configurações se necessário

In [ ]:
# Exibe configuração atual
Config.print_config()

# Para alterar configurações:
# Config.AUDIO_FORMAT = 'mp3'
# Config.AUDIO_QUALITY = 320
# Config.DELAY_MIN = 15
# Config.DELAY_MAX = 25

## 🎯 Download com Metadados Completos (NOVO!)

Sistema aprimorado com:
- Extração completa de metadados (18+ campos)
- CSV consolidado automático
- Skip inteligente de duplicados
- Suporte a 6 formatos de áudio

In [ ]:
from downloaders import YouTubeDownloader, MetadataManager

# ========== CONFIGURAÇÕES ==========
url = "COLE_O_LINK_AQUI"  # Pode ser: vídeo, playlist ou canal
audio_format = "mp3"  # Opções: mp3, wav, flac, m4a, ogg, opus

# Altera formato temporariamente (opcional)
from config import Config
Config.AUDIO_FORMAT = audio_format

# ========== INICIALIZA ==========
downloader = YouTubeDownloader()
metadata_manager = MetadataManager()

print(f"🎵 Formato de áudio: {audio_format}")
print(f"📂 Destino: {Config.get_base_path()}")
print(f"💾 CSV: {Config.get_metadata_csv_path()}")
print("="*60)

# ========== PROCESSA DOWNLOAD ==========
result = downloader.download_from_url(url)

# ========== ESTATÍSTICAS ==========
if result['success']:
    stats = result.get('stats', downloader.get_stats())
    print(f"\n✅ Download completo!")
    print(f"Total processado: {stats['total_attempted']}")
    print(f"Sucesso: {stats['successful']}")
    print(f"Pulados: {stats['skipped']}")
    print(f"Falhas: {stats['failed']}")
    
    # Resumo do CSV
    print("\n📊 RESUMO DO CSV DE METADADOS:")
    summary = metadata_manager.get_summary()
    print(f"Total no banco: {summary['total_videos']} vídeos")
    print(f"Tamanho total: {summary['total_size_formatted']}")
    print(f"Duração total: {summary['total_duration_formatted']}")
else:
    print(f"❌ Erro: {result.get('error', 'Desconhecido')}")

## 📊 Visualizar e Exportar Metadados

In [ ]:
from downloaders import MetadataManager
import pandas as pd

# Inicializa gerenciador
metadata_manager = MetadataManager()

# Exibe resumo completo
metadata_manager.print_summary()

# Carrega CSV como DataFrame pandas
df = metadata_manager.load_csv()

print(f"\n📋 PREVIEW DO CSV ({len(df)} registros):")
print("="*60)

if not df.empty:
    # Exibe primeiras linhas com colunas principais
    display_columns = ['id', 'title', 'uploader', 'duration', 'view_count', 'download_date']
    available_columns = [col for col in display_columns if col in df.columns]
    print(df[available_columns].head(10))
    
    print(f"\n💡 Dica: Use 'df' para análises personalizadas com pandas!")
    print(f"   Exemplo: df[df['uploader'] == 'NomeDoCanal']")
else:
    print("CSV vazio - nenhum download realizado ainda")

# ========== EXPORTAR FILTRADO (OPCIONAL) ==========
# Exemplo: exportar apenas playlists
# metadata_manager.export_filtered({'download_type': 'playlist'}, 'playlists_only.csv')

## Download - Vídeo Individual

In [ ]:
from downloaders import YouTubeDownloader

downloader = YouTubeDownloader()

# Substitua pela URL do vídeo
url = "https://www.youtube.com/watch?v=VIDEO_ID"

result = downloader.download_from_url(url)

if result['success']:
    print("Download concluído!")
    print(f"Arquivo: {result.get('file', 'N/A')}")
else:
    print(f"Erro: {result.get('error', 'Desconhecido')}")

## Download - Playlist

In [ ]:
# Substitua pela URL da playlist
playlist_url = "https://www.youtube.com/playlist?list=PLAYLIST_ID"

result = downloader.download_from_url(playlist_url)

if result['success']:
    stats = result['stats']
    print(f"Playlist concluída!")
    print(f"Total: {stats['total_attempted']}")
    print(f"Sucesso: {stats['successful']}")
    print(f"Falhas: {stats['failed']}")
    print(f"Pulados: {stats['skipped']}")

## Download - Canal

In [ ]:
# Substitua pela URL do canal
channel_url = "https://www.youtube.com/@username"

result = downloader.download_from_url(channel_url)

if result['success']:
    stats = result['stats']
    print(f"Canal concluído!")
    print(f"Sucesso: {stats['successful']}")

## Download - Arquivo TXT

In [ ]:
# Cria arquivo txt de exemplo
with open('urls.txt', 'w') as f:
    f.write("https://www.youtube.com/watch?v=VIDEO_ID_1\n")
    f.write("https://www.youtube.com/watch?v=VIDEO_ID_2\n")
    f.write("https://www.youtube.com/watch?v=VIDEO_ID_3\n")

# Download
result = downloader.download_from_txt('urls.txt')

if result['success']:
    print(f"TXT concluído! ID: txt_{result['txt_id']}")
    stats = result['stats']
    print(f"Sucesso: {stats['successful']}/{result['total_urls']}")

## Estatísticas e Resumo

In [ ]:
# Resumo dos downloads
summary = drive_manager.get_download_summary()

if summary['available']:
    print("📊 RESUMO GERAL")
    print("="*50)
    print(f"Total de pastas: {summary['total_folders']}")
    print(f"Total de arquivos: {summary['total_files']}")
    print(f"Tamanho total: {summary['total_size_formatted']}")
    print("\n📁 POR TIPO:")
    print(f"  Vídeos: {summary['by_type']['video']}")
    print(f"  Playlists: {summary['by_type']['playlist']}")
    print(f"  Canais: {summary['by_type']['channel']}")
    print(f"  TXT: {summary['by_type']['txt']}")
else:
    print(f"Erro: {summary.get('error', 'Desconhecido')}")

## Limpeza (Opcional)

In [ ]:
# Remove pastas vazias
removed = drive_manager.cleanup_empty_folders()
print(f"Pastas vazias removidas: {removed}")

## Verificar Logs

In [ ]:
# Lista arquivos de log
log_path = Config.get_log_path()

if log_path.exists():
    print("📝 LOGS DISPONÍVEIS:")
    for log_file in log_path.glob('*.log'):
        size = log_file.stat().st_size / 1024
        print(f"  {log_file.name} ({size:.2f} KB)")
else:
    print("Nenhum log encontrado")